# Data Exploration Snapshot

This notebook profiles **all available monthly parquet files** to support presentation slides (combined size, schema checks, and a sample preview).

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Resolve project root from common notebook working directories.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
project_root = next((p for p in candidate_roots if (p / "README.md").exists() and (p / "src").exists()), cwd)

# Collect all monthly parquet files from common project locations.
data_dirs = [project_root / "data" / "raw", project_root / "data"]
parquet_files = []
for d in data_dirs:
    if d.exists():
        parquet_files.extend(sorted(d.glob("data-*.parquet")))

# De-duplicate files while preserving order.
seen = set()
unique_files = []
for f in parquet_files:
    r = f.resolve()
    if r not in seen:
        seen.add(r)
        unique_files.append(f)

if not unique_files:
    # Fallback: search recursively under data/ in case files are nested differently.
    unique_files = sorted((project_root / "data").rglob("data-*.parquet"))

if not unique_files:
    raise FileNotFoundError(
        f"No monthly parquet files found under {project_root / 'data'} (including subfolders)."
    )

print(f"✅ Project root: {project_root}")
print(f"✅ Found {len(unique_files)} parquet file(s)")
for f in unique_files:
    print(f"  - {f.relative_to(project_root)}")

# Open metadata for each file without loading full datasets into RAM.
meta_rows = 0
schema_reference = None
for f in unique_files:
    pf = pq.ParquetFile(f)
    meta_rows += pf.metadata.num_rows
    if schema_reference is None:
        schema_reference = pf.schema.names

print(f"\nTotal rows across all files: {meta_rows:,}")
print(f"Columns in schema: {len(schema_reference)}")
print("\n--- Column names ---")
print(schema_reference)

In [ ]:
# Build a presentation-friendly sample from all files (small row batch per file).
# This avoids loading complete parquet files into memory.
sample_parts = []
for file_path in unique_files:
    pf = pq.ParquetFile(file_path)
    first_batch = next(pf.iter_batches(batch_size=5), None)
    if first_batch is None:
        continue
    part = first_batch.to_pandas()
    part["source_file"] = file_path.name
    sample_parts.append(part)

if not sample_parts:
    raise ValueError("No rows were read from discovered parquet files.")

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df.head(15)